In [ ]:
# Finalspark header
import numpy as np
import time
from datetime import datetime, timedelta, timezone
from IPython.display import clear_output
from neuroplatform import StimShape, StimParam , IntanSofware, Trigger, Database, StimPolarity, Experiment

import pandas as pd
from dateutil import parser
from lets_plot import *
from tqdm import tqdm
LetsPlot.setup_html()

In [ ]:
# Establish API credentials
token = "<insert token here>"
exp = Experiment(token)
print(f'Electrodes: {exp.electrodes}') # Electrodes that you can used


In [ ]:
# Attribute list for Stim obj
print(StimParam().__doc__)

## Electrode channel setup

Pairs of electrodes on the same organoid are intended use here. One for network entrainment, one for arbritrary data encoding. Below is the setup for a 3-organoid experiment, chosen based on >=95% Activity via baseline

In [ ]:
phase1_params, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
phase1_params = StimParam()
phase1_params.index = exp.electrodes[8]
phase1_params.trigger_key = 0
phase1_params.phase_amplitude1 = 5
phase1_params.phase_amplitude2 = 5
phase1_params.display_attributes()
print(last_updated_paramaters)

In [ ]:
phase2_params, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
phase2_params = StimParam()
phase2_params.index = exp.electrodes[17]
phase2_params.trigger_key = 1
phase2_params.phase_amplitude1 = 5
phase2_params.phase_amplitude2 = 5
phase2_params.display_attributes()
print(last_updated_paramaters)

In [ ]:
phase3_params, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
phase3_params = StimParam()
phase3_params.index = exp.electrodes[27]
phase3_params.trigger_key = 2
phase3_params.phase_amplitude1 = 5
phase3_params.phase_amplitude2 = 5
phase3_params.display_attributes()
print(last_updated_paramaters)

In [ ]:
training_elec_1, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
training_elec_1 = StimParam()
training_elec_1.index = exp.electrodes[15]
training_elec_1.trigger_key = 3
training_elec_1.phase_amplitude1 = 10
training_elec_1.phase_amplitude2 = 10
training_elec_1.display_attributes()
print(last_updated_paramaters)

In [ ]:
training_elec_2, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
training_elec_2 = StimParam()
training_elec_2.index = exp.electrodes[23]
training_elec_2.trigger_key = 4
training_elec_2.phase_amplitude1 = 10
training_elec_2.phase_amplitude2 = 10
training_elec_2.display_attributes()
print(last_updated_paramaters)

In [ ]:
training_elec_3, last_updated_paramaters = exp.best_stim_param(exp.electrodes[0])
training_elec_3 = StimParam()
training_elec_3.index = exp.electrodes[30]
training_elec_3.trigger_key = 5
training_elec_3.phase_amplitude1 = 10
training_elec_3.phase_amplitude2 = 10
training_elec_3.display_attributes()
print(last_updated_paramaters)

In [ ]:
intan = IntanSofware()
trigger_gen = Trigger()

stim_param_list = [phase1_params, phase2_params, phase3_params, training_elec_1, training_elec_2, training_elec_3]
phaseShiftSwitchTimestamps = []

intan.set_count([0])

try:
    if exp.start(): # Signal the start of an experiment to all users
        # Measure impedance
        intan.impedance()

        # Disable Variation STD (keep a fixed threshold)
        intan.var_threshold(False)

        intan.send_stimparam(stim_param_list)
        start_exp = datetime.utcnow()

        # Init trigger arrays
        organoid_1_entrainment = np.zeros(16, dtype=np.uint8)  
        organoid_2_entrainment = np.zeros(16, dtype=np.uint8) 
        organoid_2_entrainment = np.zeros(16, dtype=np.uint8)  

        # Adjacent electrodes for training
        organoid_1_training = np.zeros(16, dtype=np.uint8)  
        organoid_2_training = np.zeros(16, dtype=np.uint8)  
        organoid_2_training = np.zeros(16, dtype=np.uint8) 

        # Set up triggers for each organoid (natural frequency and adjacent for training)
        organoid_1_entrainment[0] = 1  # Organoid 1 natural 
        organoid_1_training[3] = 1  # Organoid 1 training 

        organoid_2_entrainment[1] = 1  # Organoid 2 natural
        organoid_2_training[4] = 1  # Organoid 2 training 

        organoid_2_entrainment[2] = 1  # Organoid 3 natural 
        organoid_2_training[5] = 1  # Organoid 3 training 

        # Derive natural frequencies based on empirical data (in Hz), which corresponds to the inverse of the sleep interval (in seconds)
        # These can be dynamically derived from baseline recordings (FFT) or literature-based assumptions
        natural_frequencies_hz = {
            "organoid_1": 4, 
            "organoid_2": 1,  
            "organoid_3": 0.125, 
        }

        # Convert frequencies into sleep intervals (in seconds)
        natural_intervals = {key: 1.0 / freq for key, freq in natural_frequencies_hz.items()}

        # Training frequency overlay (e.g., 50 Hz for all organoids)
        training_frequency_hz = 30
        training_interval = 1.0 / training_frequency_hz  # Training overlay interval (in seconds)

        # ------------------------------------------------
        # Baseline Recording: No stimulation (5 mins)
        # ------------------------------------------------
        print(f"Starting Baseline recording at {datetime.utcnow()}")
        time.sleep(300)  # 5 minutes of passive baseline recording

        # Store phase shift timestamp for transition
        current_time = datetime.utcnow()
        phaseShiftSwitchTimestamps.append(current_time)

        # Dynamic loop for each organoid's phase space, with alternating passive and training phases

        def stimulate_phase(organoid, trigger_natural, trigger_train, interval_natural, duration=2.5, training=False):
            """
            Stimulates a neuronal organoid using a natural-frequency protocol, with a flag enabled training stimuli

                    Parameters:
                        :parameter organoid (str): Identifier for the organoid, used for logging or debugging.
                        :parameter trigger_natural (Any): Trigger signal or command for natural-frequency stimulation.
                        :parameter trigger_train (Any): Trigger signal or command for training overlay stimulation (used if training is True).
                        :parameter (float): Interval (in seconds) between natural-frequency pulses.
                        :parameter (float): Total duration (in minutes) for which the phase will run.
                        :parameter (bool): If True, overlays a higher-frequency training stimulation alongside the natural pulses.

                    Behavior:
                        Sends natural-frequency pulses at half the interval (to approximate a continuous rhythm).
                        If `training` flag is True, sends additional training pulses between natural pulses, at a separate interval defined globally as `training_interval`.
            """
            end_time = datetime.utcnow() + timedelta(minutes=duration)
            
            while datetime.utcnow() < end_time:
                # Entrainment stimulation / Natural Rhythm freq.
                trigger_gen.send(trigger_natural)
                time.sleep(interval_natural / 2)
                
                if training:
                    # Overlay training data stream - same organoid
                    trigger_gen.send(trigger_train)
                    time.sleep(training_interval)

        def phase_shift(trigger, start_freq, target_freq, duration=30, steps=50):
            """
            Gradually shift frequency from start_freq to target over a given duration.

                    Parameters:
                        :parameter trigger (Any): Trigger signal or command for natural-frequency stimulation.
                        :parameter start_freq (Any): Start frequency (in Hz).
                        :parameter target_freq (Any): Target frequency (in Hz).
                        :parameter duration (Any): Duration (in minutes).
                        :parameter steps (Any): Number of steps to shift.

                    Behavior:
                        Shifts the stimulation by `steps` steps, starting at `start_freq` and ending at `target_freq`.
            """
            start_sleep = 1.0 / start_freq
            end_sleep = 1.0 / target_freq
            
            # Loop through the number of steps, gradually changing the frequency
            for i in range(steps):
                # Interpolate the sleep time
                current_sleep = start_sleep + (end_sleep - start_sleep) * (i / steps)
                trigger_gen.send(trigger)
                time.sleep(current_sleep)
            
            print(f"Phase shift completed: transitioned from {start_freq} Hz to {target_freq} Hz")

        # ------------------------------------------------
        # Phase Space 1: Organoid 1 Passive (2.5 mins) - Natural frequency stimulation
        # ------------------------------------------------
        print(f"Starting Phase 1 Passive (Organoid 1, natural frequency) at {datetime.utcnow()}")
        stimulate_phase("Organoid 1", organoid_1_entrainment, organoid_1_training, natural_intervals["organoid_1"], training=False)

        # Store phase shift timestamp for transition
        current_time = datetime.utcnow()
        phaseShiftSwitchTimestamps.append(current_time)

        # -----------------------------------------------
        # Phase Space 1: Organoid 1 Training (2.5 mins) - Overlay data stream on top of natural frequency
        # ------------------------------------------------
        print(f"Starting Phase 1 Training (Organoid 1) at {datetime.utcnow()}")
        stimulate_phase("Organoid 1", organoid_1_entrainment, organoid_1_training, natural_intervals["organoid_1"], training=True)

        # Store phase shift timestamp for transition
        current_time = datetime.utcnow()
        phaseShiftSwitchTimestamps.append(current_time)

        # ------------------------------------------------
        # Phase Shift: Dynamic transition after Phase 1 Training
        # ------------------------------------------------
            
        phase_shift(organoid_1_entrainment, start_freq=2.0, end_freq=1.5, steps=50)

        # Store phase shift timestamp
        current_time = datetime.utcnow()
        phaseShiftSwitchTimestamps.append(current_time)

        # ------------------------------------------------
        # Phase Space 2: Organoid 2 Passive (2.5 mins) - Natural frequency stimulation
        # ------------------------------------------------
        print(f"Starting Phase 2 Passive (Organoid 2, natural frequency) at {datetime.utcnow()}")
        stimulate_phase("Organoid 2", organoid_2_entrainment, organoid_2_training, natural_intervals["organoid_2"], training=False)

        # Store phase shift timestamp for transition
        current_time = datetime.utcnow()
        phaseShiftSwitchTimestamps.append(current_time)

        # ------------------------------------------------
        # Phase Space 2: Organoid 2 Training (2.5 mins) - Overlay data stream on top of natural frequency
        # ------------------------------------------------
        print(f"Starting Phase 2 Training (Organoid 2) at {datetime.utcnow()}")
        stimulate_phase("Organoid 2", organoid_2_entrainment, organoid_2_training, natural_intervals["organoid_2"], training=True)

        # Store phase shift timestamp for transition
        current_time = datetime.utcnow()
        phaseShiftSwitchTimestamps.append(current_time)

        # ------------------------------------------------
        # Phase Shift: Dynamic transition after Phase 2 Training
        # ------------------------------------------------
            
        phase_shift(organoid_2_entrainment, start_freq=1.5, end_freq=1.0, steps=50)

        # ------------------------------------------------
        # Phase Space 3: Organoid 3 Passive (2.5 mins) - Natural frequency stimulation
        # ------------------------------------------------
        print(f"Starting Phase 3 Passive (Organoid 3, natural frequency) at {datetime.utcnow()}")
        stimulate_phase("Organoid 3", organoid_2_entrainment, organoid_2_training, natural_intervals["organoid_3"], training=False)

        # Store phase shift timestamp for transition
        current_time = datetime.utcnow()
        phaseShiftSwitchTimestamps.append(current_time)

        # ------------------------------------------------
        # Phase Space 3: Organoid 3 Training (2.5 mins) - Overlay data stream on top of natural frequency
        # ------------------------------------------------
        print(f"Starting Phase 3 Training (Organoid 3) at {datetime.utcnow()}")
        stimulate_phase("Organoid 3", organoid_2_entrainment, organoid_2_training, natural_intervals["organoid_3"], training=True)

        # Store phase shift timestamp for transition
        current_time = datetime.utcnow()
        phaseShiftSwitchTimestamps.append(current_time)

        # ------------------------------------------------
        # Interim Recording: No stimulation (5 mins)
        # ------------------------------------------------
        print(f"Starting Interim (Passive) recording at {datetime.utcnow()}")
        time.sleep(300)  # 5 minutes of passive baseline recording

        # Store phase shift timestamp for transition
        current_time = datetime.utcnow()
        phaseShiftSwitchTimestamps.append(current_time)
        
        # ------------------------------------------------
        # Phase Space 1: Organoid 1 Passive (2.5 mins)
        # ------------------------------------------------
        print(f"Starting Phase 1 Passive (Organoid 1) at {datetime.utcnow()}")
        stimulate_phase("Organoid 1", organoid_1_entrainment, organoid_1_training, natural_intervals["organoid_1"], training=False)

        # Store phase shift timestamp for transition
        current_time = datetime.utcnow()
        phaseShiftSwitchTimestamps.append(current_time)

        # ------------------------------------------------
        # Phase Shift: Dynamic transition after PhaseSpace 1 Passive
        # ------------------------------------------------
            
        phase_shift(organoid_1_entrainment, start_freq=2.0, end_freq=1.5, steps=50)
        phaseShiftSwitchTimestamps.append(current_time)
        
        # ------------------------------------------------
        # Phase Space 2: Organoid 1 Passive (2.5 mins)
        # ------------------------------------------------
        print(f"Starting Phase 2 Passive (Organoid 2) at {datetime.utcnow()}")
        stimulate_phase("Organoid 2", organoid_2_entrainment, organoid_2_training, natural_intervals["organoid_2"], training=False)

        # Store phase shift timestamp for transition
        current_time = datetime.utcnow()
        phaseShiftSwitchTimestamps.append(current_time)

        # ------------------------------------------------
        # Phase Shift: Dynamic transition after PhaseSpace 2 Passive
        # ------------------------------------------------
            
        phase_shift(organoid_2_entrainment, start_freq=1.5, end_freq=1.0, steps=50)
        phaseShiftSwitchTimestamps.append(current_time)
        
        # ------------------------------------------------
        # Phase Space 3: Organoid 1 Passive (2.5 mins)
        # ------------------------------------------------
        print(f"Starting Phase 3 Passive (Organoid 3) at {datetime.utcnow()}")
        stimulate_phase("Organoid 3", organoid_2_entrainment, organoid_2_training, natural_intervals["organoid_3"], training=False)

        # Store phase shift timestamp for transition
        current_time = datetime.utcnow()
        phaseShiftSwitchTimestamps.append(current_time)

        # ------------------------------------------------
        # Post-Stimulation Phase: Record the network behavior after all stimulations (5 mins)
        # ------------------------------------------------
        print(f"Starting Post-Stimulation recording at {datetime.utcnow()}")
        time.sleep(300)
        # Record post-stimulation behavior for 5 minutes
        
        # ------------------------------------------------
        # Disable all stimulations after the experiment
        # ------------------------------------------------
        for stim in stim_param_list:
            stim.enable = False
        intan.send_stimparam(stim_param_list)

        # Experiment ends
        stop_exp = datetime.utcnow()
        print(f"Experiment completed at {stop_exp}. Phase shift timestamps: {phaseShiftSwitchTimestamps}")

finally:
    # Close the connection to trigger generator
    trigger_gen.close()
    # Enable variation threshold again
    intan.var_threshold(True)
    # Close the connection to intan software
    intan.close()
    # Signal the end of an experiment to all users
    exp.stop()



In [ ]:
# Collect timestamps
print(phaseShiftSwitchTimestamps)

print(start_exp)

print(stop_exp)

In [ ]:
# Load DB libs

import pandas as pd
from dateutil import parser
from lets_plot import *
from tqdm import tqdm
LetsPlot.setup_html()

In [ ]:
# Establish db connection and get exp name
db = Database()
exp_name = exp.exp_name
exp_name

In [ ]:
# Get current time func
def get_current_utc_time():
    """Returns the current UTC time as a datetime object."""
    return datetime.now(timezone.utc)

In [ ]:
# Start time parser - Timestamps gained from experiment protocols
s1 = datetime(2024, 5, 2, 9, 0, 0, 941232)
s1

# s1 = get_current_utc_time() // ALTERNATIVE FORMAT
s1 = parser.parse('2024-06-19T10:35:00.000Z')
s1

# End time parser
s2 = datetime(2024, 5, 2, 12, 0, 0, 956840)
s2

In [ ]:
# Trigger data collection point
triggers = db.get_all_triggers(s1, s2)
triggers

In [ ]:
# SPM data collection point
df_spike_count = db.get_spike_count(s1, s2, exp_name)
df_spike_count

In [ ]:
# Raw spike data collection point
df_spike_event = db.get_spike_event(s1,s2,exp_name)
df_spike_event